# Monitoring Notebook

Builds `bda_gold.brand_monitoring` from the dense monthly Gold result.
Includes share-change explanation and alert testing.

## Alert Rule
Alert when Brand#32's share in any category falls by at least 5 percentage points
compared with the previous complete consecutive month.

Boundary months (first and last) are excluded as potentially incomplete.
The production alert query is in `sql/brand_share_alert.sql`.

In [0]:
%sql
-- observed coverage boundaries (assumption: boundary months may be partial)
SELECT
    MIN(order_date) AS observed_min_order_date,
    MAX(order_date) AS observed_max_order_date,
    CAST(date_trunc('MONTH', MIN(order_date)) AS DATE) AS coverage_min_month,
    CAST(date_trunc('MONTH', MAX(order_date)) AS DATE) AS coverage_max_month,
    COUNT(DISTINCT CAST(date_trunc('MONTH', order_date) AS DATE)) AS month_count
FROM bda_gold.brand_sales;

In [0]:
%sql
-- build from dense monthly result, not recomputed from brand_sales
-- boundary months (first and last) excluded as potentially incomplete
CREATE OR REPLACE TABLE bda_gold.brand_monitoring AS
WITH date_bounds AS (
    SELECT
        CAST(date_trunc('MONTH', MIN(order_date)) AS DATE) AS min_month,
        CAST(date_trunc('MONTH', MAX(order_date)) AS DATE) AS max_month
    FROM bda_gold.brand_sales
),
with_lag AS (
    SELECT
        m.month_start,
        m.brand,
        m.category,
        m.brand_revenue,
        m.category_revenue,
        m.market_share,
        LAG(m.market_share) OVER (
            PARTITION BY m.brand, m.category ORDER BY m.month_start
        ) AS previous_market_share,
        LAG(m.month_start) OVER (
            PARTITION BY m.brand, m.category ORDER BY m.month_start
        ) AS previous_month_start
    FROM bda_gold.brand_category_monthly m
)
SELECT
    w.month_start,
    w.brand,
    w.category,
    w.brand_revenue,
    w.category_revenue,
    w.market_share,
    w.previous_market_share,
    (w.market_share - w.previous_market_share) * 100 AS share_change_percentage_points,
    CASE
        WHEN w.month_start > (SELECT min_month FROM date_bounds)
         AND w.month_start < (SELECT max_month FROM date_bounds)
        THEN true ELSE false
    END AS is_complete_month,
    CASE
        WHEN w.month_start > (SELECT min_month FROM date_bounds)
         AND w.month_start < (SELECT max_month FROM date_bounds)
         AND ADD_MONTHS(w.month_start, -1) > (SELECT min_month FROM date_bounds)
         AND w.previous_market_share IS NOT NULL
         AND w.previous_month_start = ADD_MONTHS(w.month_start, -1)
        THEN true ELSE false
    END AS is_comparable_month
FROM with_lag w;

In [0]:
%sql
-- Coverage report: brand-category-months with zero revenue
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN brand_revenue = 0 THEN 1 ELSE 0 END) AS zero_revenue_rows,
    SUM(CASE WHEN market_share IS NULL THEN 1 ELSE 0 END) AS null_share_rows,
    SUM(CASE WHEN is_complete_month THEN 1 ELSE 0 END) AS complete_months,
    SUM(CASE WHEN is_comparable_month THEN 1 ELSE 0 END) AS comparable_months,
    COUNT(DISTINCT month_start) AS distinct_months
FROM bda_gold.brand_monitoring;

-- Latest 20 rows for Brand#32
SELECT
    month_start, category,
    ROUND(brand_revenue, 2) AS brand_revenue,
    ROUND(category_revenue, 2) AS category_revenue,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(previous_market_share * 100, 2) AS prev_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_complete_month, is_comparable_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
ORDER BY month_start DESC, category
LIMIT 20;

In [0]:
%sql
-- Latest comparable month's Brand#32 shares and changes
SELECT
    month_start, category,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_comparable_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
  AND is_comparable_month
  AND month_start = (
      SELECT MAX(month_start)
      FROM bda_gold.brand_monitoring
      WHERE is_comparable_month AND brand = 'Brand#32'
  )
ORDER BY category;

In [0]:
%sql
-- arithmetic decomposition of share change for comparable brand-category-months
-- b0=prev brand rev, b1=curr brand rev, c0=prev cat rev, c1=curr cat rev, s0=b0/c0
-- brand_effect = 100*(b1-b0)/c1
-- category_effect = -100*s0*(c1-c0)/c1
-- sum = 100*(b1/c1 - b0/c0)
CREATE OR REPLACE TABLE bda_gold.brand_share_change_explanation AS
WITH with_lag AS (
    SELECT
        month_start, brand, category,
        brand_revenue, category_revenue,
        market_share, previous_market_share,
        is_comparable_month,
        LAG(brand_revenue) OVER (PARTITION BY brand, category ORDER BY month_start) AS prev_brand_rev,
        LAG(category_revenue) OVER (PARTITION BY brand, category ORDER BY month_start) AS prev_cat_rev
    FROM bda_gold.brand_monitoring
)
SELECT
    month_start, brand, category,
    brand_revenue, prev_brand_rev,
    category_revenue, prev_cat_rev,
    market_share, previous_market_share,
    100 * (brand_revenue - prev_brand_rev) / category_revenue AS brand_revenue_effect_pp,
    -100 * previous_market_share * (category_revenue - prev_cat_rev) / category_revenue AS category_revenue_effect_pp,
    100 * (brand_revenue - prev_brand_rev) / category_revenue
        + (-100 * previous_market_share * (category_revenue - prev_cat_rev) / category_revenue) AS total_effect_pp,
    100 * (market_share - previous_market_share) AS actual_share_change_pp,
    ABS(
        100 * (brand_revenue - prev_brand_rev) / category_revenue
        + (-100 * previous_market_share * (category_revenue - prev_cat_rev) / category_revenue)
        - 100 * (market_share - previous_market_share)
    ) AS identity_error_pp
FROM with_lag
WHERE is_comparable_month;

In [0]:
%sql
-- Brand#32 latest comparable share-change explanation
SELECT
    month_start, category,
    ROUND(brand_revenue_effect_pp, 4) AS brand_effect_pp,
    ROUND(category_revenue_effect_pp, 4) AS category_effect_pp,
    ROUND(total_effect_pp, 4) AS total_effect_pp,
    ROUND(actual_share_change_pp, 4) AS actual_change_pp,
    ROUND(identity_error_pp, 8) AS identity_error_pp
FROM bda_gold.brand_share_change_explanation
WHERE brand = 'Brand#32'
  AND month_start = (
      SELECT MAX(month_start)
      FROM bda_gold.brand_share_change_explanation
      WHERE brand = 'Brand#32'
  )
ORDER BY category;

In [0]:
%sql
-- validate decomposition identity across all rows
SELECT
    COUNT(*) AS total_rows,
    MAX(identity_error_pp) AS max_identity_error_pp,
    CASE WHEN MAX(identity_error_pp) < 0.001 THEN 'PASS' ELSE 'FAIL' END AS identity_check
FROM bda_gold.brand_share_change_explanation;

## Alert Demonstration

Synthetic monitoring data tests all alert scenarios.
Real Gold tables are not modified.

In [0]:
# synthetic alert testing: 7 scenarios using temp views only
latest_months = spark.sql("""
    SELECT DISTINCT month_start
    FROM bda_gold.brand_monitoring
    WHERE is_comparable_month
    ORDER BY month_start DESC
    LIMIT 2
""").collect()

latest_month = latest_months[0][0]
prev_month = latest_months[1][0]

def run_alert(view_name):
    return spark.sql(f"""
        WITH latest AS (
            SELECT MAX(month_start) AS month_start
            FROM {view_name}
            WHERE is_comparable_month
        )
        SELECT COUNT(*) AS affected_category_count
        FROM {view_name} m
        JOIN latest ON m.month_start = latest.month_start
        WHERE m.brand = 'Brand#32'
          AND m.share_change_percentage_points <= -5
    """).collect()[0][0]

def make_demo(view_name, current_share, prev_share=0.10,
              comparable=True, complete=True, missing_baseline=False):
    cs = 'true' if comparable else 'false'
    cm = 'true' if complete else 'false'
    change = (current_share - prev_share) * 100
    if missing_baseline:
        spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW {view_name} AS
            SELECT
                CAST('{latest_month}' AS DATE) AS month_start,
                'Brand#32' AS brand, 'TEST_CAT' AS category,
                {current_share * 1000.0} AS brand_revenue, 1000.0 AS category_revenue,
                {current_share} AS market_share,
                NULL AS previous_market_share,
                {change} AS share_change_percentage_points,
                true AS is_complete_month,
                false AS is_comparable_month
        """)
    else:
        spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW {view_name} AS
            SELECT
                CAST('{prev_month}' AS DATE) AS month_start,
                'Brand#32' AS brand, 'TEST_CAT' AS category,
                {prev_share * 1000.0} AS brand_revenue, 1000.0 AS category_revenue,
                {prev_share} AS market_share,
                NULL AS previous_market_share,
                NULL AS share_change_percentage_points,
                true AS is_complete_month,
                false AS is_comparable_month
            UNION ALL
            SELECT
                CAST('{latest_month}' AS DATE) AS month_start,
                'Brand#32' AS brand, 'TEST_CAT' AS category,
                {current_share * 1000.0} AS brand_revenue, 1000.0 AS category_revenue,
                {current_share} AS market_share,
                {prev_share} AS previous_market_share,
                {change} AS share_change_percentage_points,
                {cm} AS is_complete_month,
                {cs} AS is_comparable_month
        """)

tests = [
    ("0pp change",       "demo_0pp",        0.10,   0.10, True,  True,  False, 0),
    ("-4.99pp",          "demo_499",        0.0501, 0.10, True,  True,  False, 0),
    ("-5.00pp",          "demo_500",        0.05,   0.10, True,  True,  False, 1),
    ("-6.00pp",          "demo_600",        0.04,   0.10, True,  True,  False, 1),
    ("+6pp",             "demo_pos6",       0.16,   0.10, True,  True,  False, 0),
    ("incomplete month",  "demo_incomplete", 0.04,   0.10, False, False, False, 0),
    ("missing baseline",  "demo_missing",    0.04,   0.10, False, True,  True,  0),
]

for label, view, cur, prev, comp, cmplt, mb, expected in tests:
    make_demo(view, cur, prev, comparable=comp, complete=cmplt, missing_baseline=mb)
    n = run_alert(view)
    assert n == expected, f"{label}: expected {expected}, got {n}"
    print(f"PASS  {label}: affected_category_count = {n}")

print("\nAll 7 alert scenarios passed.")